# 03. Feature Engineering Pipeline
**AI Customer Support Ticket Intelligence Platform**

This notebook focuses on preparing raw inputs for Machine Learning. We combine text columns (`Ticket_Subject` and `Ticket_Description`), apply string cleaning, perform TF-IDF vectorization, encode target priority and categories, and split the data into training and test folds.

In [ ]:
import pandas as pd
import numpy as np
import re
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

cleaned_csv_path = '../dataset/cleaned_customer_support_tickets.csv'
df = pd.read_csv(cleaned_csv_path)
df.head(2)

### 1. Text Preprocessing Pipeline
Combine `Ticket_Subject` and `Ticket_Description` into a single text block, strip punctuations, lowercase all words, and clean brackets/slashes.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    # Remove product placeholders, codes, punctuations
    text = re.sub(r'\{.*?\}', '', text)  # remove {product_purchased} etc
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Combine text fields
df['Combined_Text'] = df['Ticket_Subject'].fillna('') + ' ' + df['Ticket_Description'].fillna('')
df['Cleaned_Text'] = df['Combined_Text'].apply(clean_text)

print("Sample raw text:")
print(df['Combined_Text'].iloc[0][:150])
print("\nSample cleaned text:")
print(df['Cleaned_Text'].iloc[0][:150])

### 2. TF-IDF Text Vectorization

In [ ]:
vectorizer = TfidfVectorizer(max_features=2500, stop_words='english', ngram_range=(1, 2))
X_tfidf = vectorizer.fit_transform(df['Cleaned_Text'])
print(f"TF-IDF Matrix shape: {X_tfidf.shape}")

# Save the vectorizer for prediction serving
joblib.dump(vectorizer, '../models/vectorizer.pkl')
print("Vectorizer saved to models/vectorizer.pkl")

### 3. Encode Labels for Targets
* Target 1: `Priority` (Low, Medium, High)
* Target 2: `Ticket_Category` (Technical, Billing, Refund, Shipping, Account, Login)

In [ ]:
le_priority = LabelEncoder()
y_priority = le_priority.fit_transform(df['Priority'])
joblib.dump(le_priority, '../models/priority_encoder.pkl')
print("Priority labels mapping:", dict(zip(le_priority.classes_, le_priority.transform(le_priority.classes_))))

le_category = LabelEncoder()
y_category = le_category.fit_transform(df['Ticket_Category'])
joblib.dump(le_category, '../models/category_encoder.pkl')
print("Category labels mapping:", dict(zip(le_category.classes_, le_category.transform(le_category.classes_))))

### 4. Create Train-Test Splits

In [ ]:
# Split for Priority Model
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_tfidf, y_priority, test_size=0.2, random_state=42)

# Split for Category Model
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_tfidf, y_category, test_size=0.2, random_state=42)

# Save matrices for models notebook to load
np.savez('../models/ml_split_data.npz', 
         y_train_p=y_train_p, y_test_p=y_test_p,
         y_train_c=y_train_c, y_test_c=y_test_c)
joblib.dump(X_train_p, '../models/X_train_p.pkl')
joblib.dump(X_test_p, '../models/X_test_p.pkl')
joblib.dump(X_train_c, '../models/X_train_c.pkl')
joblib.dump(X_test_c, '../models/X_test_c.pkl')

print("Splits completed and stored.")